# Buffer-stock model (Jappelli, Padula, Pistaferri) + оценка theta


In [ ]:
# !pip install numpy pandas matplotlib scipy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import brentq

plt.style.use("seaborn-v0_8-whitegrid")
np.random.seed(42)


In [ ]:
R = 1.025
G = 1.015
p_zero = 0.005
rho = 2.0
b = 1.02

sigma_N = 0.16
sigma_V = 0.28

n_households = 1000
T = 100

beta_low, beta_high = 0.86, 0.96
beta_i = np.random.uniform(beta_low, beta_high, size=n_households)


In [ ]:
def draw_lognormal_mean1(sigma, size):
    mu = -0.5 * sigma**2
    return np.random.lognormal(mean=mu, sigma=sigma, size=size)

def draw_transitory_with_zero_prob(size, sigma=sigma_V, p0=p_zero):
    v = draw_lognormal_mean1(sigma, size)
    zero_mask = np.random.rand(size) < p0
    v[zero_mask] = 0.0
    return v

def kappa_lower(beta, R=R, rho=rho):
    return 1.0 - (1.0 / R) * (R * beta) ** (1.0 / rho)

def kappa_upper(beta, R=R, rho=rho, p0=p_zero):
    return 1.0 - (1.0 / R) * (p0 * R * beta) ** (1.0 / rho)

def c_func_approx(x, beta, b=b):
    k_low = kappa_lower(beta)
    k_high = kappa_upper(beta)
    term = x - (1.0 / b) * (np.log1p(np.exp(b * x)) - np.log(2.0))
    c = 2.0 * (k_high - k_low) * term + k_low * x
    return np.clip(c, 0.0, x)

def next_x(x_t, c_t, N_tp1, V_tp1, R=R, G=G):
    return (R * (x_t - c_t) / (G * N_tp1)) + V_tp1


In [ ]:
def expected_drift(x, beta, n_mc=4000):
    N_draw = draw_lognormal_mean1(sigma_N, n_mc)
    V_draw = draw_transitory_with_zero_prob(n_mc)
    c = c_func_approx(x, beta)
    x_next = next_x(x, c, N_draw, V_draw)
    return np.mean(x_next - x)

def solve_target_x(beta, xmin=1e-4, xmax=30.0):
    f = lambda x: expected_drift(x, beta)
    grid = np.linspace(xmin, xmax, 120)
    vals = np.array([f(g) for g in grid])
    signs = np.sign(vals)
    idx = np.where(np.diff(signs) != 0)[0]
    if len(idx) == 0:
        return grid[np.argmin(np.abs(vals))]
    i = idx[0]
    return brentq(f, grid[i], grid[i + 1])

beta_grid = np.linspace(beta_low, beta_high, 35)
x_star_grid = np.array([solve_target_x(b) for b in beta_grid])
x_star_i = np.interp(beta_i, beta_grid, x_star_grid)
print(f"x* (median): {np.median(x_star_i):.3f}")


In [ ]:
x_t = draw_transitory_with_zero_prob(n_households)
theta_t = []
records_rep = []
rep = 0

for t in range(T):
    c_t = c_func_approx(x_t, beta_i)
    gap_t = x_t - x_star_i
    cov = np.cov(c_t, gap_t, ddof=0)[0, 1]
    var = np.var(gap_t)
    theta_t.append(cov / var)

    records_rep.append({"t": t, "x": x_t[rep], "c": c_t[rep]})

    N_tp1 = draw_lognormal_mean1(sigma_N, n_households)
    V_tp1 = draw_transitory_with_zero_prob(n_households)
    x_t = next_x(x_t, c_t, N_tp1, V_tp1)

theta_hat = float(np.mean(theta_t))
print(f"Estimated theta = {theta_hat:.4f}")
print("Reference (wp150 baseline): theta ≈ 0.3995")


In [ ]:
x_plot = np.linspace(0, 12, 400)
beta_med = float(np.median(beta_i))
c_plot = c_func_approx(x_plot, beta_med)

plt.figure(figsize=(8, 5))
plt.plot(x_plot, c_plot, lw=2, label='c(x), approximation')
plt.plot(x_plot, x_plot, '--', lw=1, label='45° line')
plt.title('Approximate consumption function (Padula)')
plt.xlabel('x')
plt.ylabel('c')
plt.legend()
plt.show()


In [ ]:
rep_df = pd.DataFrame(records_rep)
plt.figure(figsize=(10, 5))
plt.plot(rep_df['t'], rep_df['x'], lw=1.6, label='x_t')
plt.plot(rep_df['t'], rep_df['c'], lw=1.6, label='c_t')
plt.title('Representative household: x_t and c_t over time')
plt.xlabel('Period')
plt.ylabel('Ratio')
plt.legend()
plt.show()


In [ ]:
theta_series = pd.Series(theta_t)
plt.figure(figsize=(10, 4.5))
plt.plot(theta_series.index, theta_series.values, marker='o', ms=3, lw=1)
plt.axhline(theta_hat, color='red', linestyle='--', label=f'mean theta = {theta_hat:.4f}')
plt.axhline(0.3995, color='black', linestyle=':', label='wp150 reference = 0.3995')
plt.title('Covariance ratio by period')
plt.xlabel('Period')
plt.ylabel('Theta_t')
plt.legend()
plt.show()


In [ ]:
x_t = draw_transitory_with_zero_prob(n_households)
for t in range(T):
    c_t = c_func_approx(x_t, beta_i)
    N_tp1 = draw_lognormal_mean1(sigma_N, n_households)
    V_tp1 = draw_transitory_with_zero_prob(n_households)
    x_t = next_x(x_t, c_t, N_tp1, V_tp1)

final_gap = x_t - x_star_i
plt.figure(figsize=(8, 4.8))
plt.hist(final_gap, bins=40, alpha=0.85)
plt.title('Distribution of wealth gaps at end of simulation')
plt.xlabel('x_T - x*')
plt.ylabel('Households')
plt.show()
